## Problem: Why gradients become unstable in deep networks?

Let's consider a very simple neural network. Each layer just multiplies its input by a weight:

```text
h1 = w1 * x
h2 = w2 * h1
h3 = w3 * h2
...
hn = wn * h(n-1)
```

During backpropagation.

```text
L is Loss function

dL/dh2 = dL/dh3 * w3

dL/dh1 = dL/dh2 * w2
       = dL/dh3 * w3 * w2
       
```

**The farther back we go, the more weights get multiplied together.**

For the earliest layer, the gradient can involve a long product:

`wn * w(n-1) * ... * w3 * w2`


- If all these weights have absolute values smaller than 1 → vanishing gradients.

- If all these weights have absolute values larger than 1 → exploding gradients.


What's more
```text
dL/dw2 = dL/dh2 * h1

dL/dh2 = dL/dh3 * w3
h1 = w1 * x

dL/dw2 = (dL/dh3 * w3) * (w1 * x)
```

So weights gradient is influenced by **forward** (activation) and **backward** (gradient passed back).


## How to init weight?

Note:

```text
dL/dw2 = dL/dh2 * h1

dL/dh2 = dL/dh3 * w3
h1 = w1 * x

dL/dw2 = (dL/dh3 * w3) * (w1 * x)
```

So weights gradient is influenced by **forward** (activation) and **backward** (gradient passed back).

When init weight, need to consider both direction!

### If indepandent inputs sample from N(0, 1), variance will add up, std become sqrt(n)

> Problem:
>
> Even with just one Linear layer, the output activation scale grows as the number of inputs increases.

In [2]:
import torch

torch.manual_seed(0)

for n in [16, 256, 1024]:
    # 4096 smaples to make result stable
    x = torch.randn(4096, n)
    w = torch.randn(n)
    y = x @ w
    print(f"n={n:4d} | input std={x.std():.2f} | output std={y.std():.2f}")

n=  16 | input std=1.00 | output std=2.75
n= 256 | input std=1.00 | output std=15.15
n=1024 | input std=1.00 | output std=31.76


## An init strategy example: scale down according to n

Ex. PyTorch nn.Linear

uses something called 

> torch.nn.init.kaiming_uniform_(weight, a=math.sqrt(5))

Details doesn't matter, the idea is: variance grows with the number of inputs, we scale down the initial weights based on fan_in to counteract this growth.

This only deal with fan_in.

>TODO: Xavier -> both in and out.
>
>TODO: it's not 1 trcik fits all, how to init depends on later structure 

## Residual

### What is residual?

> without residual:
> y1 = f(x)
>
> with residual
> y2 = x + f(x)

### Why it's useful?

say 

> y1 = w1 * x

it needs to learn everything from scratch by only using w1. Say w1 = 0.1 or 100, it could cause unstable gradient.

now

> y2 = w1 * x + x

it just needs to learn **what to change**, and if output is still y2 = 0.1x or 100x, we know network adjust w1 (-0.9 or 99) to achieve that, it's what it wants!

## RMSNorm

### What are we looking for?

> forward pass looks like this
>
> input -> box_1 -> box_2 -> .... -> box_n
>
> each black box could be anything (ex. MLP, multi-head attention etc)

magnitude of activations can affect the gradients of trainable weights.

So we want to keep activation (a.k.a output) scales (a.k.a how large) under control — neither too large nor too small.

**Our goal: Normalize the activation vector so that its RMS is approximately 1, while preserving the ratios between its elements.**

### How?

Say the current layer's RMS is x, Divide everything by x and then the new RMS = 1. I will skip the proof, pretty simple.

### why add a small epsilon?

Avoid divide by 0.

In [2]:
import torch

def rms_norm(x, eps=1e-8):
    return x / torch.sqrt(torch.mean(x**2) + eps)

# Pytorch impl
norm = torch.nn.RMSNorm(2, eps=1e-8)

x = torch.tensor([3., 4.])

# 10x difference between input but nearly same output
print(rms_norm(x))
print(norm(x))
print(rms_norm(x * 10))
print(norm(x * 10))

tensor([0.8485, 1.1314])
tensor([0.8485, 1.1314], grad_fn=<MulBackward0>)
tensor([0.8485, 1.1314])
tensor([0.8485, 1.1314], grad_fn=<MulBackward0>)


## Why Pre-Norm?

In Pre-Norm, RMSNorm does not sit on the identity path, allowing gradients to flow directly through it.